# Entrenando el Modelo y Probandolo con Frases Nuevas

En el notebook anterior dejamos todo listo: el dataset tokenizado, separado en train y test, y el modelo con las capas base de BERT congeladas. Aqui, configuramos el entrenamiento con el `Trainer` de Hugging Face, entrenamos el modelo y lo probamos con una frase que nunca vio.

Este notebook esta pensado para correrse con GPU. En Colab, anda a Entorno de ejecucion > Cambiar tipo de entorno de ejecucion > GPU antes de empezar.

### El plan de hoy

1. Recrear rapidamente el dataset y el modelo congelado del notebook anterior (por si esta es una sesion nueva de Colab).
2. Definir una metrica de evaluacion.
3. Configurar `TrainingArguments` y el `Trainer`.
4. Entrenar y evaluar el modelo.
5. Probarlo con una frase nueva y ver la categoria ganadora con su score de confianza.
6. Bonus: entrenar tambien la version de tres categorias y comparar resultados.

## 0. Preparando el entorno

Instalamos las librerias necesarias e importamos todo. Tambien confirmamos que tenemos GPU disponible: el entrenamiento con CPU seria mucho mas lento.

Un detalle de Colab: a veces viene con una version de `torchvision` que no es compatible con la version de `torch` instalada, y eso rompe a la libreria `datasets` con un error como `ImportError: cannot import name 'VideoReader' from 'torchvision.io'`. No usamos `torchvision` en este notebook (trabajamos solo con texto, no con imagenes ni video), asi que la solucion mas simple es directamente desinstalarlo.

In [1]:
# Quitamos torchvision: no la necesitamos (solo trabajamos con texto), y en algunos entornos
# de Colab su version rompe la deteccion automatica que hace la libreria datasets
!pip uninstall -y -q torchvision
!pip install -q transformers datasets accelerate scikit-learn evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00


In [2]:
import re

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)
import evaluate

print("GPU disponible:", torch.cuda.is_available())

GPU disponible: True


## 1. Retomando lo del notebook anterior

Repetimos, en forma condensada, los pasos del notebook anterior: cargar el dataset, quedarnos con la version binaria, tokenizar, y congelar las capas base del modelo. Es el mismo codigo que ya escribimos; si te queda alguna duda de por que hacemos cada paso, podes volver al notebook anterior.

In [3]:
# Cargamos el dataset de tweets y nos quedamos solamente con las columnas que vamos a utilizar
url_dataset = "https://raw.githubusercontent.com/satyajeetkrjha/kaggle-Twitter-US-Airline-Sentiment-/master/Tweets.csv"
columnas_utiles = ["tweet_id", "airline", "text", "airline_sentiment"]
df = pd.read_csv(url_dataset, usecols=columnas_utiles)


# Renombramos las columnas para trabajar con nombres más claros en español
df = df.rename(columns={
    "tweet_id": "id",
    "airline": "aerolinea",
    "text": "texto",
    "airline_sentiment": "sentimiento_original",
})


# Definimos una función para limpiar los tweets, eliminando menciones (@usuario) y enlaces
def quitar_menciones_y_links(texto):
    texto = re.sub(r"@\w+", "", texto)
    texto = re.sub(r"http\S+", "", texto)
    return texto.strip()


# Aplicamos la limpieza a todos los tweets y guardamos el resultado en una nueva columna
df["texto_sentimiento"] = df["texto"].apply(quitar_menciones_y_links)


# Convertimos las etiquetas de sentimiento a IDs numéricos y nos quedamos solamente con las dos clases que queremos estudiar
etiqueta2id = {"negative": 0, "positive": 1}
df_binario = df[df["sentimiento_original"].isin(etiqueta2id.keys())].copy()
df_binario["label"] = df_binario["sentimiento_original"].map(etiqueta2id)


# Tomamos una muestra balanceada: la misma cantidad de tweets negativos y positivos.
# El dataset original esta desbalanceado (~80% negative, ~20% positive). Si sampleamos
# al azar sin corregir eso, el modelo puede "aprender" a predecir siempre la clase
# mayoritaria y aun asi parecer que tiene buen accuracy, sin haber aprendido nada util.
tweets_por_clase = 1500

df_muestra = (
    df_binario.groupby("label", group_keys=False)
    .apply(lambda grupo: grupo.sample(n=min(len(grupo), tweets_por_clase), random_state=42))
    .reset_index(drop=True)
)

# Dividimos en 80% para entrenamiento y 20% para prueba.
# stratify mantiene la misma proporción de clases en ambos conjuntos (ahora 50/50)
train_df, test_df = train_test_split(
    df_muestra,
    test_size=0.2,
    stratify=df_muestra["label"],
    random_state=42
)

/tmp/ipykernel_2196/3637294023.py:41: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda grupo: grupo.sample(n=min(len(grupo), tweets_por_clase), random_state=42))


In [4]:
# Cargamos el tokenizador de BERT multilingüe que utilizaremos para convertir los tweets en tokens e IDs
model_name = "bert-base-multilingual-cased"

tokenizer = AutoTokenizer.from_pretrained(model_name)


# Definimos cómo vamos a tokenizar nuestros textos: recortamos los textos demasiado largos y rellenamos con padding
# (los tensores de un batch deben tener la misma forma para poder processar !)
# los textos cortos para que todos tengan una longitud de 64 tokens
def tokenizar(batch):

    return tokenizer(
        batch["texto_sentimiento"],
        truncation=True,
        padding="max_length",
        max_length=64,
    )


# Indicamos las columnas que BERT necesita para hacer el entrenamiento
columnas_modelo = ["input_ids", "attention_mask", "labels"]


# Convertimos nuestros DataFrames de pandas en Datasets de Hugging Face para poder utilizar las herramientas de entrenamiento de Transformers
train_dataset = Dataset.from_pandas(
    train_df[["texto_sentimiento", "label"]].reset_index(drop=True)
)

test_dataset = Dataset.from_pandas(
    test_df[["texto_sentimiento", "label"]].reset_index(drop=True)
)


# Tokenizamos los dos datasets y cambiamos el nombre de "label" a "labels", que es el nombre que espera el modelo de Transformers
train_dataset = train_dataset.map(tokenizar, batched=True).rename_column("label", "labels")
test_dataset = test_dataset.map(tokenizar, batched=True).rename_column("label", "labels")


# Convertimos las columnas necesarias al formato de PyTorch para que puedan ser utilizadas directamente durante el entrenamiento
train_dataset.set_format("torch", columns=columnas_modelo)

test_dataset.set_format("torch", columns=columnas_modelo)

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

Map:   0%|          | 0/2400 [00:00<?, ? examples/s]

Map:   0%|          | 0/600 [00:00<?, ? examples/s]

In [5]:
# Definimos la correspondencia entre los IDs numéricos y las etiquetas de sentimiento que queremos que el modelo reconozca
id2label = {0: "Negativo", 1: "Positivo"}
label2id = {"Negativo": 0, "Positivo": 1}


# Cargamos BERT preparado para clasificación de textos con 2 categorías La capa de clasificación se crea para nuestra tarea y sus pesos se entrenarán
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

# Congelamos todos los parámetros de BERT para conservar sus pesos pre-entrenados. Solo la nueva capa de clasificación quedará disponible para el entrenamiento
for param in model.bert.parameters():
    param.requires_grad = False

model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### Resumen de la preparación de BERT para clasificación

En estos tres bloques hemos preparado nuestros datos y el modelo para hacer **fine-tuning con BERT**:

1. **Preparamos los datos:** cargamos los tweets, limpiamos el texto, conservamos solo las clases *negative* y *positive*, las convertimos a IDs (`0` y `1`) y dividimos los datos en entrenamiento y prueba.

2. **Preparamos los textos para BERT:** convertimos los tweets en tokens con el tokenizador de BERT. Además, hacemos que todas las secuencias tengan un máximo de 64 tokens mediante *truncation* y *padding*. Finalmente, convertimos los datos al formato que utiliza PyTorch.

3. **Preparamos el modelo:** cargamos BERT con una nueva capa de clasificación para distinguir entre **Negativo** y **Positivo**. Congelamos los parámetros de BERT (`requires_grad = False`), por lo que sus pesos pre-entrenados no cambiarán. **Solo entrenaremos la nueva capa de clasificación.**

Esta estrategia se conoce como **feature extraction**: utilizamos BERT como un generador fijo de representaciones y aprendemos un clasificador específico para nuestra tarea.


## 2. Definiendo como vamos a medir el desempeño

El `Trainer` no calcula metricas de evaluacion por si solo mas alla de la funcion de perdida (loss); tenemos que decirle explicitamente que queremos ver. Usamos la libreria `evaluate` de Hugging Face para calcular el accuracy: el porcentaje de tweets del conjunto de test que el modelo clasifico correctamente.

In [6]:
metrica_accuracy = evaluate.load("accuracy") # Cargamos la métrica de accuracy


# Función que calcula el accuracy durante la evaluación
def calcular_metricas(eval_pred):

    logits, labels = eval_pred # Separamos las predicciones del modelo (logits) y las etiquetas correctas
    predicciones = np.argmax(logits, axis=-1) # Elegimos la clase con el valor más alto entre las predicciones

    # Comparamos las predicciones con las etiquetas reales y calculamos el accuracy
    return metrica_accuracy.compute(
        predictions=predicciones,
        references=labels
    )

## 3. Configurando el entrenamiento

`TrainingArguments` reune todas las decisiones sobre como se va a entrenar el modelo: cuantas veces recorremos el dataset completo (`num_train_epochs`), cuantos ejemplos ve el modelo antes de actualizar sus pesos (`per_device_train_batch_size`), y cuando evaluamos.

Como solo estamos entrenando la cabeza de clasificacion (el resto esta congelado), no hace falta ni un learning rate muy baja ni muchas epochs para ver resultados: pocas epochs alcanzan.

In [7]:
# Configuramos los parámetros que controlarán el entrenamiento de BERT
argumentos_entrenamiento = TrainingArguments(
    output_dir="./resultados_bert",          # Carpeta donde se guardarán los resultados
    num_train_epochs=3,                      # Entrenamos el modelo durante 3 épocas
    per_device_train_batch_size=16,          # Procesamos 16 ejemplos por batch durante el entrenamiento
    per_device_eval_batch_size=16,           # Procesamos 16 ejemplos por batch durante la evaluación
    eval_strategy="epoch",                   # Evaluamos el modelo al terminar cada época
    logging_strategy="epoch",                # Mostramos información del entrenamiento al terminar cada época
    report_to="none",                        # No enviamos los resultados a plataformas externas
)

> Durante la evaluación, no calculamos gradientes ni actualizamos pesos. Por lo tanto, la evaluación generalmente requiere menos memoria, lo que nos permite procesar batch más grandes (si quisieramos).

## 4. Armando el Trainer y entrenando

El `Trainer` conecta todas las piezas: el modelo, los argumentos de entrenamiento, los datasets de train y test, y la funcion de metricas. Una vez armado, entrenar es una sola linea.

In [8]:
trainer = Trainer(
    model=model,                        # Modelo que vamos a entrenar
    args=argumentos_entrenamiento,      # Configuración del entrenamiento
    train_dataset=train_dataset,        # Datos utilizados para entrenar
    eval_dataset=test_dataset,          # Datos utilizados para evaluar
    compute_metrics=calcular_metricas,  # Función para calcular las métricas
)

# Ejecutamos el proceso de entrenamiento
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.694609,0.683952,0.550000
2,0.678457,0.672186,0.640000
3,0.670896,0.668772,0.626667


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=450, training_loss=0.6813210042317709, metrics={'train_runtime': 39.9249, 'train_samples_per_second': 180.338, 'train_steps_per_second': 11.271, 'total_flos': 236799949824000.0, 'train_loss': 0.6813210042317709, 'epoch': 3.0})

El Trainer es un class de HugginFace para facilitar el proceso de entrenamiento.   
Lo que hace el Trainer:

```
                Trainer
                   │
        ┌──────────┼──────────┐
        ▼          ▼          ▼
     batches    forward    backward
        │        pass       pass
        │          │          │
        └──────────┼──────────┘
                   ▼
             update weights
                   │
                   ▼
              evaluation
                   │
                   ▼
                metrics
```

```
- PyTorch
    ↓
Escribe tú mismo el bucle de entrenamiento
        

-  Hugging Face Transformers
     ↓
Trainer → Gestiona el ciclo de entrenamiento por ti
```

## 5. Evaluando el modelo entrenado

Una vez terminado el entrenamiento, podemos pedirle al `Trainer` que evalue el modelo sobre el conjunto de test y nos devuelva las metricas.

In [9]:
resultados = trainer.evaluate()
resultados

Training Loss,Validation Loss,Epoch,Accuracy
0.670896,0.668772,3,0.626667


{'eval_loss': 0.6687717437744141, 'eval_accuracy': 0.6266666666666667}

## 6. Inferencia con una frase nueva

El verdadero objetivo del temario: probar el modelo con una frase que nunca vio durante el entrenamiento, y ver que categoria elige junto con su score de confianza.

In [10]:
def predecir_sentimiento(texto, modelo, tokenizador):
    modelo.eval() # Ponemos el modelo en modo evaluación
    encoding = tokenizador(texto, return_tensors="pt", truncation=True, padding=True) # Tokenizamos el texto y lo convertimos en tensores de PyTorch
    encoding = {k: v.to(modelo.device) for k, v in encoding.items()} # Movemos los tensores al mismo dispositivo que el modelo (CPU o GPU)

    with torch.no_grad(): # Desactivamos el cálculo de gradientes porque solo queremos hacer una predicción, no entrenar el modelo
        salida = modelo(**encoding)

    probabilidades = torch.softmax(salida.logits, dim=-1)[0] # Convertimos los logits en probabilidades y seleccionamos las probabilidades correspondientes a este texto
    id_ganador = int(torch.argmax(probabilidades)) # Buscamos la clase con la probabilidad más alta

    return modelo.config.id2label[id_ganador], float(probabilidades[id_ganador]) # Devolvemos el nombre de la clase y su probabilidad


In [11]:
frase_nueva = "The staff was incredibly helpful and the flight left right on time."

categoria, score = predecir_sentimiento(frase_nueva, model, tokenizer)
print(f"Frase: {frase_nueva}")
print(f"Categoria ganadora: {categoria}")
print(f"Score de confianza: {score:.4f}")

Frase: The staff was incredibly helpful and the flight left right on time.
Categoria ganadora: Positivo
Score de confianza: 0.5010


Proba con tus propias frases, incluso en español, ya que el modelo es multilingue. Fijate que pasa con frases ambiguas o mixtas (algo positivo y algo negativo en la misma oracion): son un buen ejemplo de los limites de un modelo entrenado con pocos datos y pocas epochs.

## 7. Bonus: comparando con la version de tres categorias

En el notebook anterior dejamos preparada una version del dataset con las tres categorias originales (positive, neutral, negative). Entrenemosla tambien, de la misma forma, y comparemos el accuracy final contra la version binaria.

In [ ]:
etiqueta2id_tres = {"negative": 0, "neutral": 1, "positive": 2}
id2label_tres = {0: "Negativo", 1: "Neutral", 2: "Positivo"}
label2id_tres = {"Negativo": 0, "Neutral": 1, "Positivo": 2}

df_tres = df.copy()
df_tres["label"] = df_tres["sentimiento_original"].map(etiqueta2id_tres)

# Igual que en la version binaria: muestra balanceada, esta vez por clase entre las tres categorias.
tweets_por_clase_tres = 1000

df_muestra_tres = (
    df_tres.groupby("label", group_keys=False)
    .apply(lambda grupo: grupo.sample(n=min(len(grupo), tweets_por_clase_tres), random_state=42))
    .reset_index(drop=True)
)

train_df_tres, test_df_tres = train_test_split(
    df_muestra_tres, test_size=0.2, stratify=df_muestra_tres["label"], random_state=42
)

train_dataset_tres = Dataset.from_pandas(train_df_tres[["texto_sentimiento", "label"]].reset_index(drop=True))
test_dataset_tres = Dataset.from_pandas(test_df_tres[["texto_sentimiento", "label"]].reset_index(drop=True))

train_dataset_tres = train_dataset_tres.map(tokenizar, batched=True).rename_column("label", "labels")
test_dataset_tres = test_dataset_tres.map(tokenizar, batched=True).rename_column("label", "labels")

train_dataset_tres.set_format("torch", columns=columnas_modelo)
test_dataset_tres.set_format("torch", columns=columnas_modelo)

model_tres = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=3, id2label=id2label_tres, label2id=label2id_tres
)

for param in model_tres.bert.parameters():
    param.requires_grad = False

In [ ]:
argumentos_entrenamiento_tres = TrainingArguments(
    output_dir="./resultados_bert_tres",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    logging_strategy="epoch",
    report_to="none",
)

trainer_tres = Trainer(
    model=model_tres,
    args=argumentos_entrenamiento_tres,
    train_dataset=train_dataset_tres,
    eval_dataset=test_dataset_tres,
    compute_metrics=calcular_metricas,
)

trainer_tres.train()
resultados_tres = trainer_tres.evaluate()

In [ ]:
print(f"Accuracy version binaria (2 categorias):   {resultados['eval_accuracy']:.4f}")
print(f"Accuracy version completa (3 categorias):   {resultados_tres['eval_accuracy']:.4f}")

Es esperable que el accuracy de la version binaria sea mas alto: distinguir entre solo dos opciones es un problema mas facil que distinguir entre tres, y ademas la categoria "neutral" suele ser la mas dificil de reconocer (muchas veces ni las personas se ponen de acuerdo en que tweet es neutral). Esto no significa que la version de tres categorias sea "peor": es un problema distinto, y en un caso real la eleccion depende de que necesita el negocio.

## Cierre: dos estrategias, y algunos errores comunes a evitar

Hoy usamos **feature extraction**: congelamos toda la base de BERT y entrenamos solo la cabeza de clasificacion. Es rapido y funciona bien con datasets chicos como el nuestro. La alternativa es el **full fine-tuning**: entrenar tambien las capas base, sin congelarlas. Suele dar mejores resultados, pero necesita mas datos, mas tiempo de computo, y una tasa de aprendizaje bien baja (algo como 2x10⁻⁵) para evitar el olvido catastrofico que mencionamos antes.

Algunos errores comunes a tener en cuenta cuando hagas esto con tus propios datos:

- **Data leakage**: evaluar el modelo con datos que tambien uso para entrenar. Por eso separamos train y test desde el principio y nunca los mezclamos.
- **Overfitting en datasets chicos**: si el dataset es muy chico y se hace full fine-tuning, el modelo puede terminar memorizando los ejemplos en vez de aprender un patron general.
- **Tokenizador y modelo desparejados**: el tokenizador tiene que ser siempre el mismo que se uso para pre-entrenar el modelo. Mezclar el tokenizador de un modelo con los pesos de otro rompe todo silenciosamente.

## Resumen

- Definimos una metrica de evaluacion (accuracy) con la libreria `evaluate`.
- Configuramos `TrainingArguments` y armamos el `Trainer`, uniendo el modelo, los datos y la metrica.
- Entrenamos el modelo y lo evaluamos sobre el conjunto de test.
- Hicimos inferencia sobre una frase nueva, mostrando la categoria ganadora y su score de confianza.
- Comparamos la version binaria contra la version de tres categorias, y entendimos por que sus resultados no son directamente comparables.
- Repasamos la diferencia entre feature extraction y full fine-tuning, y algunos errores comunes del proceso.

Con esto completamos las cuatro fases del temario de esta semana: configuracion del entorno y el modelo, preparacion del dataset, transfer learning (congelado de capas) y entrenamiento con inferencia final.